# Build the DiFF test set

Crops the faces of DiFF's official test split (Cheng et al., ACM MM 2024) with the same RetinaFace pipeline as `build_dataset.ipynb`, and uploads them as a **private** Hugging Face dataset. The lab never trains on DiFF: these crops are the test side of the `cross_diff` protocol, the public benchmark for diffusion-generated and diffusion-edited faces (subsets T2I, I2I, FS and FE).

**Access:** the synthesized images are public on the authors' Google Drive (see the [DiFF repository](https://github.com/xaCheng1996/DiFF)); this notebook downloads the 13 test zips from there, one at a time, and deletes each after cropping. The pristine images are granted per researcher through the authors' form, for academic research only and without redistribution; the Drive id of the archive in their email goes in the Colab secret `DIFF_PRISTINE_ID`, never in this repository. Its `DiFF_real/test` folder holds the pristine test images (`diff.parse_pristine_member`); they go to their own private dataset, `PRISTINE_DATASET`, so either side can be rebuilt alone, and `score_diffusion_benchmarks.ipynb` loads both.

**Runtime:** Colab Pro with a GPU and about 5 GB of free disk per zip. **Needs** the Colab secret `HF_TOKEN` with write access, for the upload cell.

### Configuration

In [ ]:
REPO_SOURCE = "git+https://github.com/paulohenriquevn/face-decision-lab@workspace"
HF_DATASET = "paulohenriquevn/face-decision-diff-test"  # private
BUILD_DIR = "/content/build_diff"
ZIP_DIR = "/content/diff_zips"
# Google Drive file ids of the test zips in the authors' public folder (test/<subset>/<method>.zip),
# as listed in October 2026.
TEST_ZIPS = {
    "T2I/Midjourney.zip": "1--CWfk8lotkSjh6pri9PZEHVn7kw_Wfe",
    "T2I/SDXL.zip": "1cSVbYpAN56u-ozC_Pkp1NH4Fj44KZVzS",
    "T2I/FreeDoM_T.zip": "14MvFLCanc91tSaHf8Xldg8E3hzXU2DVV",
    "T2I/HPS.zip": "1xUEhBwA4d6jngN526x0TFfb0RY552uAQ",
    "I2I/SDXL_Refine.zip": "1Fr5E4r-4gjLKY76ObitIZ2uxI6ytMs7r",
    "I2I/LoRA.zip": "1FFWO5XGxn1GeJok1RoDWbF7eeEQOlIqE",
    "I2I/DreamBooth.zip": "1iXoaqTmVP60X3OTz1PvGjvSkeb0dVyYB",
    "I2I/FreeDoM_I.zip": "1N-RcXROf3VWa-bQq37NYTT9Z6bk-Mp8-",
    "FS/DiffFace.zip": "1JZZnmuxM5g3Bh00wjnlmYUsli90We6f0",
    "FS/DCFace.zip": "1rPNZ3RWKni-Et7nFgKfvrX-mWvhcMIgy",
    "FE/Imagic.zip": "1UPr8RzsMcWp-qtVWwbj4WZN08OdZ4ppN",
    "FE/CoDiff.zip": "1uhWlx_bwreBdAiw-a_acjmY_4McTwR5D",
    "FE/cycle_diff.zip": "1cyZ1QPoKFFimqLmi2yvHLwwPC7aIjX8s",
}
PRISTINE_DATASET = "paulohenriquevn/face-decision-diff-pristine-test"  # private
PRISTINE_BUILD_DIR = "/content/build_diff_pristine"
# Which side to build: the attack side has been on the Hub since October 2026.
BUILD_ATTACKS = False
BUILD_PRISTINE = True

### Installation

Same as `build_dataset.ipynb`: `onnxruntime-gpu` and the ONNX backend, so RetinaFace runs on the GPU.

In [ ]:
%%capture
!pip install -q "face-decision-lab[build] @ {REPO_SOURCE}" gdown
!pip uninstall -y -q onnxruntime
!pip install -q onnxruntime-gpu

In [ ]:
import os
os.environ["DEEPFACE_BACKEND_ENGINE"] = "onnx"

import onnxruntime as ort
providers = ort.get_available_providers()
print("ONNX Runtime providers:", providers)
assert "CUDAExecutionProvider" in providers, "onnxruntime-gpu cannot see the GPU; RetinaFace would run on the CPU"

### Build the attack side

Resumable, like the main build: a rerun skips every image already in `processed.jsonl`. Each zip is deleted after cropping, so the disk never holds more than one.

In [ ]:
from pathlib import Path
import gdown
from tqdm.auto import tqdm
from face_decision_lab.build import DatasetBuilder, zip_members
from face_decision_lab.faces import RetinaFaceDetector
from face_decision_lab.sources import diff

detector = RetinaFaceDetector()
builders = {}
Path(ZIP_DIR).mkdir(parents = True, exist_ok = True)
for name, file_id in tqdm(TEST_ZIPS.items() if BUILD_ATTACKS else (), desc = "zips"):
    builder = builders.setdefault(HF_DATASET, DatasetBuilder(BUILD_DIR, detector))
    path = Path(ZIP_DIR) / name.replace("/", "_")
    if gdown.download(id = file_id, output = str(path), quiet = True, resume = True) is None:
        raise RuntimeError(f"download of {name} (Drive id {file_id}) failed; the shared file may be over its quota")
    added = builder.add_images(zip_members(path, diff.parse_member))
    builder.flush()
    path.unlink()
    print(f"{name}: {added} images")
print({repo: len(b.samples) for repo, b in builders.items()}, "crops")

### Build the bona fide side

Downloads the pristine archive by its Drive id and crops the `DiFF_real/test` images; the train and val folders are skipped.

In [ ]:
if BUILD_PRISTINE:
    from google.colab import userdata
    path = Path(ZIP_DIR) / "DiFF_real.zip"
    if gdown.download(id = userdata.get("DIFF_PRISTINE_ID"), output = str(path), quiet = True, resume = True) is None:
        raise RuntimeError("download of the pristine archive failed; the shared file may be over its quota")
    builder = builders.setdefault(PRISTINE_DATASET, DatasetBuilder(PRISTINE_BUILD_DIR, detector))
    builder.add_images(zip_members(path, diff.parse_pristine_member))
    builder.flush()
    path.unlink()
    print(len(builder.samples), "bona fide crops")

### Check detection outcomes and framing

A detector that drops one generator more often than the others biases the test, so compare the saved share per generator, the pristine images included. DiFF images are already tight face crops, so the lab's 1.5x square crop pads them with black; the grid shows how much.

In [ ]:
for builder in builders.values():
    for generator in sorted(builder.report.counts):
        counts = builder.report.counts[generator]
        print(f"{generator:24s} saved {builder.report.saved_share(generator):6.1%}  {counts}")

In [ ]:
from PIL import Image
from IPython.display import display

first = {}
for builder in builders.values():
    for sample in builder.samples:
        first.setdefault(sample.generator, builder.output_dir / sample.image_path)
grid = Image.new("RGB", (128 * len(first), 128))
for column, path in enumerate(first.values()):
    grid.paste(Image.open(path).resize((128, 128)), (128 * column, 0))
print(list(first))
display(grid)

### Upload as a private dataset

In [ ]:
from google.colab import userdata
from huggingface_hub import login
from face_decision_lab.hub import push_build

login(token = userdata.get("HF_TOKEN"))
for repo, builder in builders.items():
    print(push_build(builder.output_dir, repo))